# SCIPE SP26 Project 

## Data Wrangling

In [4]:
import json
import pandas as pd
import os
from collections import defaultdict

In [5]:
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

sample_geoid = list(data.keys())[0]
sample = data[sample_geoid]

print("Total GEOIDs:", len(data))
print("Sample GEOID:", sample_geoid)
print("Top-level keys:", list(sample.keys()))
print("Type:", sample.get("type"))
print("Datasets:", list(sample["metrics"].keys()))

Total GEOIDs: 1158
Sample GEOID: 5003
Top-level keys: ['type', 'name', 'block_group', 'census_tract', 'county', 'state', 'population', 'metrics']
Type: hawaiian_homeland
Datasets: ['2022_census_hawaiian_homelands']


In [ ]:
# Flatten nested JSON into a long-form DataFrame
rows = []
for geoid, area_data in data.items():
    base = {
        "geoid":        geoid,
        "type":         area_data.get("type"),
        "name":         area_data.get("name"),
        "block_group":  area_data.get("block_group"),
        "census_tract": area_data.get("census_tract"),
        "county":       area_data.get("county"),
        "state":        area_data.get("state"),
        "population":   area_data.get("population")
    }
    for dataset_name, metrics in area_data.get("metrics", {}).items():
        for metric_name, values in metrics.items():
            rows.append({**base, "dataset": dataset_name,
                         "metric": metric_name,
                         "absolute": values.get("absolute"),
                         "proportion": values.get("proportion")})

df_long = pd.DataFrame(rows)
print("Shape:", df_long.shape)
print("\nCounts by type:")
print(df_long.groupby("type").size())

Shape: (44520, 12)

Counts by type:
type
block_group          43320
hawaiian_homeland     1200
dtype: int64


In [7]:
# Block groups and Hawaiian homelands have completely different metrics
metrics_block = set(df_long[df_long["type"] == "block_group"]["metric"].unique())
metrics_hh    = set(df_long[df_long["type"] == "hawaiian_homeland"]["metric"].unique())
shared        = metrics_block & metrics_hh

print(f"Block group metrics:       {len(metrics_block)}")
print(f"Hawaiian homeland metrics: {len(metrics_hh)}")
print(f"Shared metrics:            {len(shared)}")
print(f"\nNull absolute:   {df_long['absolute'].isna().sum()}")
print(f"Null proportion: {df_long['proportion'].isna().sum()}")

Block group metrics:       40
Hawaiian homeland metrics: 16
Shared metrics:            0

Null absolute:   2606
Null proportion: 3944


## PostgreSQL + PostGIS Setup

In [8]:
from sqlalchemy import create_engine, text

engine = create_engine("postgresql://postgres@localhost/ccsvi_db")

with engine.connect() as conn:
    print("Connected:", conn.execute(text("SELECT version();")).fetchone()[0][:40])
    print("PostGIS:",   conn.execute(text("SELECT postgis_version();")).fetchone()[0][:20])

Connected: PostgreSQL 18.3 (Postgres.app) on aarch6
PostGIS: 3.6 USE_GEOS=1 USE_P


In [ ]:
# Build geographic_areas table
geo_rows = []
for geoid, area in data.items():
    geo_rows.append({
        "geoid":        geoid,
        "type":         area.get("type"),
        "name":         area.get("name"),
        "block_group":  area.get("block_group"),
        "census_tract": area.get("census_tract"),
        "county":       area.get("county"),
        "state":        area.get("state"),
        "population":   area.get("population"),
    })

df_geo = pd.DataFrame(geo_rows)
df_geo.to_sql("geographic_areas", engine, if_exists="replace", index=False)

print(f"✓ geographic_areas loaded: {len(df_geo)} rows")
print(df_geo["type"].value_counts().to_string())

✓ geographic_areas loaded: 1158 rows
type
block_group          1083
hawaiian_homeland      75


/Users/mvchaella/anaconda3/lib/python3.12/site-packages/pandas/io/sql.py:2059: SAWarning: Did not recognize type 'geometry' of column 'geom'
  self.meta.reflect(


In [11]:
# Add geometry column
with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE geographic_areas
        ADD COLUMN IF NOT EXISTS geom geometry(Geometry, 4326);
    """))
    conn.commit()

In [12]:
# Block group boundaries
bg_path = "../public/data/2020_Census_Block_Groups_Stripped.geojson"
with open(bg_path) as f:
    bg_geojson = json.load(f)

updated, skipped = 0, 0
with engine.connect() as conn:
    for feature in bg_geojson["features"]:
        geoid     = feature["properties"].get("geoid20")
        geom_json = json.dumps(feature["geometry"])
        result = conn.execute(text("""
            UPDATE geographic_areas
            SET geom = ST_SetSRID(ST_GeomFromGeoJSON(:g), 4326)
            WHERE geoid = :id
        """), {"g": geom_json, "id": str(geoid)})
        if result.rowcount > 0: updated += 1
        else: skipped += 1
    conn.commit()

print(f"✓ Block groups — updated: {updated}, skipped: {skipped}")

✓ Block groups — updated: 1056, skipped: 0


In [13]:
# Hawaiian homeland boundaries
hhl_path = "../public/data/Census_Hawaiian_Homelands_hhl10_Stripped.geojson"
with open(hhl_path) as f:
    hhl_geojson = json.load(f)

updated, skipped = 0, 0
with engine.connect() as conn:
    for feature in hhl_geojson["features"]:
        geoid     = feature["properties"].get("GEOID10")
        geom_json = json.dumps(feature["geometry"])
        result = conn.execute(text("""
            UPDATE geographic_areas
            SET geom = ST_SetSRID(ST_GeomFromGeoJSON(:g), 4326)
            WHERE geoid = :id
        """), {"g": geom_json, "id": str(geoid)})
        if result.rowcount > 0: updated += 1
        else: skipped += 1
    conn.commit()

print(f"✓ Hawaiian homelands — updated: {updated}, skipped: {skipped}")

✓ Hawaiian homelands — updated: 73, skipped: 2


In [ ]:
# Verify geometry 
with engine.connect() as conn:
    total, with_geom = conn.execute(text("""
        SELECT COUNT(*), COUNT(geom) FROM geographic_areas;
    """)).fetchone()

print(f"Total rows:       {total}")
print(f"With geometry:    {with_geom}")
print(f"Missing geometry: {total - with_geom}")

Total rows:       1158
With geometry:    1129
Missing geometry: 29


## Vulnerability Datasets

In [15]:
data_path = "../public/data/vulnerability_datasets/"
files = sorted([f for f in os.listdir(data_path) if f.endswith(".csv")])
print(f"Found {len(files)} files:")
for f in files:
    print(" -", f)

Found 15 files:
 - 2022_census_hawaiian_homelands.csv
 - age_of_structure.csv
 - aggregate_vehicles.csv
 - genders.csv
 - health_insurance.csv
 - households_w_computer.csv
 - income_share_of_fpl.csv
 - internet_subscription.csv
 - limited_english_speaking.csv
 - living_arrangements.csv
 - person_under_5_65_females.csv
 - person_under_5_65_males.csv
 - population_group_quarters.csv
 - race_origin.csv
 - tenure.csv


In [ ]:
from functools import reduce

column_map = {
    "Total Population Under 5 (Per Capita)":              "perc_under_5",
    "Total Population Under 18 (Per Capita)":             "perc_under_18",
    "Total Population Over 65 (Per Capita)":              "perc_over_65",
    "Estimate!!Total!!INDIVIDUALS' INCOME IN THE PAST 12 MONTHS (IN 2022 INFLATION-ADJUSTED DOLLARS)!!Population 15 years and over!!Median income (dollars)": "median_income",
    "Estimate!!Total:!!No Internet access (Per Capita)":  "perc_no_internet",
    "Estimate!!Total:!!No Computer (Per Capita)":         "perc_no_computer",
    "Total Living alone (Per Capita)":                                                    "perc_living_alone",
    "Estimate!!Total:!!In households:!!Householder:!!Male:!!Living alone (Per Capita)":   "perc_male_living_alone",
    "Estimate!!Total:!!In households:!!Householder:!!Female:!!Living alone (Per Capita)": "perc_female_living_alone",
    "Total Limited English Speaking Households (Per Capita)":                                               "perc_limited_english",
    "Estimate!!Total:!!Spanish:!!Limited English speaking household (Per Capita)":                          "perc_limited_english_spanish",
    "Estimate!!Total:!!Asian and Pacific Island languages:!!Limited English speaking household (Per Capita)": "perc_limited_english_asian_pi",
    "Estimate!!Total:!!Renter occupied (Per Capita)":     "perc_renter_occupied",
    "Total Under 100% FPL (Per Capita)":                  "perc_poverty_100",
    "Total Under 150% FPL (Per Capita)":                  "perc_poverty_150",
    "Total Under 200% FPL (Per Capita)":                  "perc_poverty_200",
    "Total Housing Built Before 1990 (Per Capita)":       "perc_pre_1990_housing",
    "Estimate!!Aggregate number of vehicles available:":  "total_vehicles",
    "Estimate!!Total:!!White alone (Per Capita)":                                      "perc_white",
    "Estimate!!Total:!!Black or African American alone (Per Capita)":                  "perc_black",
    "Estimate!!Total:!!American Indian and Alaska Native alone (Per Capita)":          "perc_aian",
    "Estimate!!Total:!!Asian alone (Per Capita)":                                      "perc_asian",
    "Estimate!!Total:!!Native Hawaiian and Other Pacific Islander alone (Per Capita)":  "perc_nhpi",
    "Estimate!!Total:!!Some Other Race alone (Per Capita)":                            "perc_other_race",
    "Estimate!!Total:!!Two or More Races: (Per Capita)":                               "perc_two_or_more_races",
    "No Health Insurance Coverage (Per Capita)":          "perc_uninsured",
    "Males Under 5 (Per Capita)":    "perc_males_under_5",
    "Males Under 18 (Per Capita)":   "perc_males_under_18",
    "Males Over 65 (Per Capita)":    "perc_males_over_65",
    "Females Under 5 (Per Capita)":  "perc_females_under_5",
    "Females Under 18 (Per Capita)": "perc_females_under_18",
    "Females Over 65 (Per Capita)":  "perc_females_over_65",
    "!!Total:!!Institutionalized population: (Per Capita)":                                                "perc_institutionalized",
    "!!Total:!!Institutionalized population:!!Correctional facilities for adults (Per Capita)":            "perc_inst_correctional",
    "!!Total:!!Institutionalized population:!!Juvenile facilities (Per Capita)":                           "perc_inst_juvenile",
    "!!Total:!!Institutionalized population:!!Nursing facilities/Skilled-nursing facilities (Per Capita)": "perc_inst_nursing",
    "Estimate!!Total:!!Male: (Per Capita)":   "perc_male",
    "Estimate!!Total:!!Female: (Per Capita)": "perc_female",
}

dfs = []

for file in files:
    df = pd.read_csv(os.path.join(data_path, file))
    df["geoid"] = df["Geography"].str.split("US").str[-1]

    for col in df.columns:
        if col in column_map:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    cols_to_keep = ["geoid"] + [c for c in df.columns if c in column_map]
    if len(cols_to_keep) == 1:
        print(f"⚠️  {file}: no matching columns, skipping")
        continue

    df_filtered = df[cols_to_keep].rename(columns=column_map)
    matched = [c for c in df_filtered.columns if c != "geoid"]
    print(f"✓ {file}: {matched}")
    dfs.append(df_filtered)

df_wide = reduce(lambda left, right: pd.merge(left, right, on="geoid", how="outer"), dfs)

print(f"\nFinal shape: {df_wide.shape}")
print(f"Total GEOIDs: {df_wide['geoid'].nunique()}")

df_wide.to_sql("vulnerability_wide", engine, if_exists="replace", index=False)

✓ 2022_census_hawaiian_homelands.csv: ['median_income', 'perc_under_5', 'perc_under_18', 'perc_over_65']
✓ age_of_structure.csv: ['perc_pre_1990_housing']
✓ aggregate_vehicles.csv: ['total_vehicles']
✓ genders.csv: ['perc_male', 'perc_female']
✓ health_insurance.csv: ['perc_uninsured']
✓ households_w_computer.csv: ['perc_no_computer']
✓ income_share_of_fpl.csv: ['perc_poverty_100', 'perc_poverty_150', 'perc_poverty_200']
✓ internet_subscription.csv: ['perc_no_internet']
✓ limited_english_speaking.csv: ['perc_limited_english', 'perc_limited_english_spanish', 'perc_limited_english_asian_pi']
✓ living_arrangements.csv: ['perc_living_alone', 'perc_male_living_alone', 'perc_female_living_alone']
✓ person_under_5_65_females.csv: ['perc_females_under_5', 'perc_females_under_18', 'perc_females_over_65']
✓ person_under_5_65_males.csv: ['perc_males_under_5', 'perc_males_under_18', 'perc_males_over_65']
✓ population_group_quarters.csv: ['perc_institutionalized', 'perc_inst_correctional', 'perc_in

320

In [ ]:
# whoops
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS vulnerability_wide;"))
    conn.commit()
    print("✓ Old table dropped")

df_wide.to_sql("vulnerability_wide", engine, if_exists="replace", index=False)
print(f"✓ Written: {len(df_wide)} rows, {len(df_wide.columns)} columns")

✓ Old table dropped
✓ Written: 1158 rows, 39 columns


In [27]:
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT
            COUNT(*)                AS total_rows,
            COUNT(perc_uninsured)   AS has_insurance,
            COUNT(perc_poverty_100) AS has_poverty,
            COUNT(perc_no_internet) AS has_internet,
            COUNT(median_income)    AS has_income
        FROM vulnerability_wide;
    """)).fetchone()

print(f"Total rows:         {row[0]}")
print(f"Has insurance data: {row[1]}")
print(f"Has poverty data:   {row[2]}")
print(f"Has internet data:  {row[3]}")
print(f"Has income data:    {row[4]}")

Total rows:         1158
Has insurance data: 1050
Has poverty data:   1050
Has internet data:  1050
Has income data:    42


## Hazard Layers

In [ ]:
import json

hazard_path = "../public/data/Hazards/"

with open(hazard_path + "Flood_Hazard_trimmed.geojson") as f:
    flood = json.load(f)

print("Total features:", len(flood["features"]))
print("\nSample properties:")
print(flood["features"][0]["properties"])
print("\nGeometry type:", flood["features"][0]["geometry"]["type"])

Total features: 2624

Sample properties:
{'fld_zone': 'VE', 'static_bfe': 15.0, 'depth': -9999}

Geometry type: MultiPolygon


In [29]:
with open(hazard_path + "filtered_slr_cstl_erosn_0pt5ft.geojson") as f:
    slr = json.load(f)

print("Total features:", len(slr["features"]))
print("\nSample properties:")
print(slr["features"][0]["properties"])
print("\nGeometry type:", slr["features"][0]["geometry"]["type"])

Total features: 325

Sample properties:
{'id': 0.0, 'Shape_Leng': 11109.2247206, 'Shape_Area': 42218.8974052}

Geometry type: MultiPolygon


In [ ]:
import os
from sqlalchemy import text

hazard_path = "../public/data/Hazards/"

hazard_files = {
    "Flood_Hazard_trimmed.geojson":              ("flood",                  None),
    "filtered_slr_cstl_erosn_0pt5ft.geojson":    ("erosion",                "0pt5ft"),
    "filtered_slr_cstl_erosn_1pt1ft.geojson":    ("erosion",                "1pt1ft"),
    "filtered_slr_cstl_erosn_2pt0ft.geojson":    ("erosion",                "2pt0ft"),
    "filtered_slr_cstl_erosn_3pt2ft.geojson":    ("erosion",                "3pt2ft"),
    "filtered_slr_exposure_area_0pt5ft.geojson":  ("exposure_area",          "0pt5ft"),
    "filtered_slr_exposure_area_1pt1ft.geojson":  ("exposure_area",          "1pt1ft"),
    "filtered_slr_exposure_area_2pt0ft.geojson":  ("exposure_area",          "2pt0ft"),
    "filtered_slr_exposure_area_3pt2ft.geojson":  ("exposure_area",          "3pt2ft"),
    "filtered_slr_passive_fld_0pt5ft.geojson":    ("passive_flood",          "0pt5ft"),
    "filtered_slr_passive_fld_1pt1ft.geojson":    ("passive_flood",          "1pt1ft"),
    "filtered_slr_passive_fld_2pt0ft.geojson":    ("passive_flood",          "2pt0ft"),
    "filtered_slr_passive_fld_3pt2ft.geojson":    ("passive_flood",          "3pt2ft"),
    "filtered_slr_potent_econ_loss_0pt5ft.geojson":("potential_economic_loss","0pt5ft"),
    "filtered_slr_potent_econ_loss_1pt1ft.geojson":("potential_economic_loss","1pt1ft"),
    "filtered_slr_potent_econ_loss_2pt0ft.geojson":("potential_economic_loss","2pt0ft"),
    "filtered_slr_potent_econ_loss_3pt2ft.geojson":("potential_economic_loss","3pt2ft"),
    "filtered_slr_potent_fld_hwys_0pt5ft.geojson": ("potential_flood_highways","0pt5ft"),
    "filtered_slr_potent_fld_hwys_1pt1ft.geojson": ("potential_flood_highways","1pt1ft"),
    "filtered_slr_potent_fld_hwys_2pt0ft.geojson": ("potential_flood_highways","2pt0ft"),
    "filtered_slr_potent_fld_hwys_3pt2ft.geojson": ("potential_flood_highways","3pt2ft"),
}

In [ ]:
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hazard_zones;"))
    conn.execute(text("""
        CREATE TABLE hazard_zones (
            hazard_id    SERIAL PRIMARY KEY,
            hazard_type  TEXT,
            slr_scenario TEXT,
            fld_zone     TEXT,
            source_file  TEXT,
            geom         geometry(Geometry, 4326)
        );
    """))
    conn.commit()

✓ hazard_zones table recreated


In [ ]:
for filename, (hazard_type, slr_scenario) in hazard_files.items():
    filepath = os.path.join(hazard_path, filename)

    if not os.path.exists(filepath):
        print(f"⚠️  {filename}: file not found, skipping")
        continue

    with open(filepath) as f:
        geojson = json.load(f)

    features = geojson["features"]
    inserted = 0

    with engine.connect() as conn:
        for feature in features:
            geom_json  = json.dumps(feature["geometry"])
            props      = feature.get("properties", {})
            fld_zone   = props.get("fld_zone", None)

            conn.execute(text("""
                INSERT INTO hazard_zones (hazard_type, slr_scenario, fld_zone, source_file, geom)
                VALUES (
                    :hazard_type,
                    :slr_scenario,
                    :fld_zone,
                    :source_file,
                    ST_SetSRID(ST_GeomFromGeoJSON(:geom), 4326)
                )
            """), {
                "hazard_type":  hazard_type,
                "slr_scenario": slr_scenario,
                "fld_zone":     fld_zone,
                "source_file":  filename,
                "geom":         geom_json
            })
            inserted += 1

        conn.commit()

    print(f"✓ {filename}: {inserted} features loaded")

✓ Flood_Hazard_trimmed.geojson: 2624 features loaded
✓ filtered_slr_cstl_erosn_0pt5ft.geojson: 325 features loaded
✓ filtered_slr_cstl_erosn_1pt1ft.geojson: 339 features loaded
✓ filtered_slr_cstl_erosn_2pt0ft.geojson: 336 features loaded
✓ filtered_slr_cstl_erosn_3pt2ft.geojson: 336 features loaded
✓ filtered_slr_exposure_area_0pt5ft.geojson: 6 features loaded
✓ filtered_slr_exposure_area_1pt1ft.geojson: 6 features loaded
✓ filtered_slr_exposure_area_2pt0ft.geojson: 6 features loaded
✓ filtered_slr_exposure_area_3pt2ft.geojson: 6 features loaded
✓ filtered_slr_passive_fld_0pt5ft.geojson: 11 features loaded
✓ filtered_slr_passive_fld_1pt1ft.geojson: 11 features loaded
✓ filtered_slr_passive_fld_2pt0ft.geojson: 12 features loaded
✓ filtered_slr_passive_fld_3pt2ft.geojson: 12 features loaded
✓ filtered_slr_potent_econ_loss_0pt5ft.geojson: 29773 features loaded
✓ filtered_slr_potent_econ_loss_1pt1ft.geojson: 30906 features loaded
✓ filtered_slr_potent_econ_loss_2pt0ft.geojson: 33377 featu

In [35]:
 with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT hazard_type, slr_scenario, COUNT(*) as features
        FROM hazard_zones
        GROUP BY hazard_type, slr_scenario
        ORDER BY hazard_type, slr_scenario;
    """))
    for row in result:
        print(f"{row[0]:<30} {str(row[1]):<10} {row[2]} features")

erosion                        0pt5ft     325 features
erosion                        1pt1ft     339 features
erosion                        2pt0ft     336 features
erosion                        3pt2ft     336 features
exposure_area                  0pt5ft     6 features
exposure_area                  1pt1ft     6 features
exposure_area                  2pt0ft     6 features
exposure_area                  3pt2ft     6 features
flood                          None       2624 features
passive_flood                  0pt5ft     11 features
passive_flood                  1pt1ft     11 features
passive_flood                  2pt0ft     12 features
passive_flood                  3pt2ft     12 features
potential_economic_loss        0pt5ft     29773 features
potential_economic_loss        1pt1ft     30906 features
potential_economic_loss        2pt0ft     33377 features
potential_economic_loss        3pt2ft     37411 features
potential_flood_highways       0pt5ft     98 features
potential_floo